In [1]:
import keras
from keras import layers,models
import keras_hub
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import numpy as np
import os

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!unzip -q "drive/MyDrive/DataSets/Faces.zip" -d "/content/data"

In [6]:
data_dir = "/content/data/UTKFace"

def create_df(directory):
    files = [f for f in os.listdir(directory) if f.endswith('.jpg')]
    data = []

    for f in files:
        parts = f.split('_')
        if len(parts) >= 3:
            try:
                data.append({
                    'path': os.path.join(directory, f),
                    'age': float(parts[0]),
                    'gender': int(parts[1]),
                    'race': int(parts[2])
                })
            except (ValueError, IndexError):
                continue

    return pd.DataFrame(data)

df = create_df(data_dir)

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

print(df.head())



                                                path   age  gender  race
0  /content/data/UTKFace/27_0_2_20170119193327962...  27.0       0     2
1  /content/data/UTKFace/22_0_2_20170116165407686...  22.0       0     2
2  /content/data/UTKFace/22_1_0_20170103183738666...  22.0       1     0
3  /content/data/UTKFace/60_1_0_20170110135905294...  60.0       1     0
4  /content/data/UTKFace/41_1_0_20170109012224846...  41.0       1     0


In [7]:
# Replace your current UTKDataLoader class with this version
class UTKDataLoader(keras.utils.PyDataset):
    def __init__(self, df, batch_size=32, img_size=(224, 224), **kwargs):
        super().__init__(**kwargs)
        self.df = df
        self.batch_size = batch_size
        self.img_size = img_size

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        self.df = self.df.sample(frac=1).reset_index(drop=True)

    def __getitem__(self, idx):
        batch_df = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]

        images = np.array([
            np.array(Image.open(p).convert('RGB').resize(self.img_size))
            for p in batch_df['path']
        ], dtype="float32")

        # --- Return a LIST of targets in specific order [Age, Gender, Race] ---
        age = (batch_df['age'].values / 100.0).astype("float32")
        gender = batch_df['gender'].values.astype("float32")
        race = batch_df['race'].values.astype("int32")

        return images, (age, gender, race)


train_ds = UTKDataLoader(train_df)
val_ds = UTKDataLoader(val_df)

In [14]:
inputs = keras.Input(shape=(224,224,3))
x = layers.Rescaling(1./225)(inputs)

base_model = keras_hub.models.ResNetBackbone.from_preset("resnet_50_imagenet")
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
age_h = models.Sequential([
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, name="age_output")
    ])(x)

gender_h = models.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid", name="gender_output")
])(x)

race_h = models.Sequential([
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(5, activation="softmax", name="race_output")
])(x)

model = keras.Model(inputs=inputs,outputs=[age_h,gender_h,race_h])
model.summary()

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_10[0… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_net_backbone    │ (None, 7, 7,      │ 23,561,152 │ rescaling_2[0][0] │
│ (ResNetBackbone)    │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ res_net_backbone… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_6        │ (None, 1)         │    262,401 │ global_average_p… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_7        │ (None, 1)         │    131,201 │ global_average_p… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_8        │ (None, 5)         │    262,917 │ global_average_p… │
│ (Sequential)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,217,671 (92.38 MB)

 Trainable params: 24,164,551 (92.18 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [19]:
# Re-run your compile cell with this list format
base_model.trainable = True
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=[
        "mse",                          # Age
        "binary_crossentropy",          # Gender
        "sparse_categorical_crossentropy" # Race
    ],
    loss_weights=[2.0, 1.0, 1.0],
    metrics=["mae", "accuracy","accuracy"]
)

In [20]:

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',      # Watch the validation loss (not training loss!)
    patience=3,              # Wait for 3 epochs of no improvement before quitting
    min_delta=0.001,         # Improvement must be at least this much to "count"
    restore_best_weights=True, # VERY IMPORTANT: Undo the overfitting by going back to the best weights
    verbose=1
)

checkpoint = keras.callbacks.ModelCheckpoint(
    "best_utk_model.keras",
    monitor="val_loss",
    save_best_only=True      # Only overwrites the file if the new model is better
)
# Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)

# Save the model
model.save("face_multi_task.keras")

Epoch 1/10
630/630 ━━━━━━━━━━━━━━━━━━━━ 330s 430ms/step - loss: 2.9134 - sequential_6_loss: 0.0411 - sequential_6_mae: 0.1587 - sequential_7_accuracy: 0.5831 - sequential_7_loss: 0.8421 - sequential_8_accuracy: 0.4092 - sequential_8_loss: 1.9920 - val_loss: 2.0878 - val_sequential_6_loss: 0.0347 - val_sequential_6_mae: 0.1418 - val_sequential_7_accuracy: 0.6606 - val_sequential_7_loss: 0.6423 - val_sequential_8_accuracy: 0.4561 - val_sequential_8_loss: 1.3697
Epoch 2/10
630/630 ━━━━━━━━━━━━━━━━━━━━ 234s 371ms/step - loss: 2.0871 - sequential_6_loss: 0.0350 - sequential_6_mae: 0.1440 - sequential_7_accuracy: 0.6619 - sequential_7_loss: 0.6392 - sequential_8_accuracy: 0.4479 - sequential_8_loss: 1.3781 - val_loss: 1.8894 - val_sequential_6_loss: 0.0301 - val_sequential_6_mae: 0.1320 - val_sequential_7_accuracy: 0.7213 - val_sequential_7_loss: 0.5533 - val_sequential_8_accuracy: 0.4955 - val_sequential_8_loss: 1.2758
Epoch 3/10
630/630 ━━━━━━━━━━━━━━━━━━━━ 234s 371ms/step - loss: 1.8726 -

In [ ]:
print()